# WorldCover remapping

In this notebook we explore how to use a lookup table to convert the non-consecutive classes in our WorldCover raster into consecutive ones.

In [1]:
# Boilerplate code to automatically reload imports and to have
# access to the project sourcecode.
%load_ext autoreload
%autoreload 2
    
import sys
import pathlib
import os

sys.path.append(os.path.abspath(".."))

In [8]:
from src.io import WORLDCOVER_LABELS
import rasterio
import numpy as np

In [5]:
with rasterio.open(WORLDCOVER_LABELS) as labels_src:
    labels = labels_src.read()

In [6]:
labels

array([[[4, 4, 4, ..., 1, 1, 1],
        [4, 4, 4, ..., 1, 1, 1],
        [4, 4, 4, ..., 1, 1, 3],
        ...,
        [3, 4, 4, ..., 4, 4, 4],
        [3, 4, 9, ..., 4, 4, 4],
        [3, 4, 9, ..., 4, 4, 4]]], shape=(1, 5120, 5120), dtype=uint8)

In [12]:
nodata_value = 0
present_labels = [int(v) for v in np.unique(labels) if int(v) != nodata_value]
present_labels

[1, 2, 3, 4, 5, 6, 8, 9]

Create the remapping from these non consecutive values to a consecutive sequence:

In [15]:
mapping = {label: value for value, label in enumerate(present_labels)}
mapping

{1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 8: 6, 9: 7}

We can now create a lookup table:

In [17]:
lut = np.full(256, 255, dtype=np.uint8)
for value, label in mapping.items():
    lut[value] = label
lut

array([255,   0,   1,   2,   3,   4,   5, 255,   6,   7, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
       255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 25

Now we can quickly convert the entire label raster from the original labels to our more classic sequential ones:

In [18]:
lut[labels]

array([[[3, 3, 3, ..., 0, 0, 0],
        [3, 3, 3, ..., 0, 0, 0],
        [3, 3, 3, ..., 0, 0, 2],
        ...,
        [2, 3, 3, ..., 3, 3, 3],
        [2, 3, 7, ..., 3, 3, 3],
        [2, 3, 7, ..., 3, 3, 3]]], shape=(1, 5120, 5120), dtype=uint8)